# Ollama-only evaluation of contextual embeddings for the local XAI-RAG knowledge base

This notebook evaluates whether **contextual chunk embeddings** improve retrieval over plain chunk embeddings in the local psychology/XAI/RAG knowledge base.

The design follows the core idea of Anthropic's contextual retrieval guide: before embedding a chunk, prepend a short context statement explaining where the chunk fits in the source document. This notebook adapts that idea to a **fully local Ollama setup**.

## Research question

Does adding document-aware context to PDF chunks improve retrieval quality for a local RAG system supporting non-clinical depression/anxiety screening communication and XAI explanation generation?

## Compared retrieval conditions

1. `plain_dense`: embed the raw chunk text only.
2. `metadata_context_dense`: prepend structured metadata from the corpus file before embedding.
3. `ollama_context_dense`: ask a local Ollama LLM to write a concise per-chunk context sentence, then prepend it before embedding.
4. `plain_hybrid`: combine dense similarity with BM25 lexical retrieval.
5. `metadata_context_hybrid`: hybrid retrieval over metadata-contextualized chunks.
6. `ollama_context_hybrid`: hybrid retrieval over Ollama-contextualized chunks.

## Primary metrics

- `Recall@K`: whether at least one relevant document appears in the top K.
- `MRR@K`: how early the first relevant document appears.
- `nDCG@K`: ranking quality when multiple relevant documents exist.
- `category_accuracy@1`: whether the top result belongs to the expected expertise category.

## Important limitation

Some evaluation queries come from the same search-query taxonomy used to build the corpus. That makes this an **internal retrieval evaluation**, not a fully independent benchmark. For a paper defense, report this clearly and add a small manually written challenge set, which this notebook includes.

In [1]:
from pathlib import Path
import csv
import hashlib
import json
import math
import os
import re
import time
import urllib.request
from collections import Counter, defaultdict

import numpy as np
from pypdf import PdfReader

ROOT = Path.cwd().parent
LOG_DIR = Path.cwd()
METADATA_PATH = ROOT / "metadata" / "selected_documents.jsonl"
SEARCH_QUERIES_PATH = LOG_DIR / "search_queries_used.txt"
CACHE_DIR = LOG_DIR / "contextual_eval_cache"
CACHE_DIR.mkdir(exist_ok=True)

OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434")
EMBED_MODEL = "nomic-embed-text"
CONTEXT_MODEL = "qwen2.5:latest"

# For a fast initial run, keep these small. For paper results, increase them.
MAX_PAGES_PER_DOC = 8
MAX_CHUNKS_PER_DOC = 8
WORDS_PER_CHUNK = 260
CHUNK_OVERLAP = 60
TOP_K = 5
MAX_EMBED_CHARS = 6000

print("ROOT:", ROOT)
print("Metadata exists:", METADATA_PATH.exists())
print("Search queries exist:", SEARCH_QUERIES_PATH.exists())

ROOT: C:\Users\MossabBatal\Desktop\psychology_mas\knowledge_db
Metadata exists: True
Search queries exist: True


## Local Ollama API helpers

The evaluation uses only the local Ollama HTTP API. No cloud embeddings, no external reranker, and no external judge model are required.

In [2]:
def ollama_json(endpoint, payload, timeout=180, retries=3):
    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        f"{OLLAMA_URL}{endpoint}",
        data=data,
        headers={"Content-Type": "application/json"},
    )
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            with urllib.request.urlopen(request, timeout=timeout) as response:
                return json.loads(response.read().decode("utf-8"))
        except Exception as exc:
            last_error = exc
            body = ""
            if hasattr(exc, "read"):
                try:
                    body = exc.read().decode("utf-8", errors="replace")[:1000]
                except Exception:
                    body = ""
            print(f"Ollama call failed on attempt {attempt}/{retries}: {exc} {body}")
            time.sleep(2 * attempt)
    raise last_error

def embed_text(text, model=EMBED_MODEL):
    for max_chars in [MAX_EMBED_CHARS, 3000, 1500, 750]:
        try:
            result = ollama_json("/api/embeddings", {"model": model, "prompt": text[:max_chars]})
            return np.asarray(result["embedding"], dtype=np.float32)
        except Exception as exc:
            print(f"Embedding failed with max_chars={max_chars}: {exc}")
    raise RuntimeError("Embedding failed after all truncation fallbacks")

def generate_text(prompt, model=CONTEXT_MODEL, timeout=300):
    result = ollama_json(
        "/api/generate",
        {
            "model": model,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0},
        },
        timeout=timeout,
    )
    return " ".join(result.get("response", "").split())

print("Embedding smoke test dimension:", len(embed_text("student wellbeing retrieval test")))

Embedding smoke test dimension: 768


## Corpus loading

We evaluate only documents marked `downloaded` in `selected_documents.jsonl` and only when the PDF exists locally. Each chunk keeps its `doc_id`, category, title, page number, source URL, and text.

In [3]:
def load_documents():
    docs = []
    with METADATA_PATH.open("r", encoding="utf-8") as handle:
        for line in handle:
            doc = json.loads(line)
            if doc.get("download_status") != "downloaded":
                continue
            pdf_path = ROOT / doc.get("local_pdf_path", "")
            if not pdf_path.exists():
                continue
            doc["abs_pdf_path"] = str(pdf_path)
            docs.append(doc)
    return sorted(docs, key=lambda item: item["doc_id"])

docs = load_documents()
print("Downloaded/evaluable documents:", len(docs))
Counter(doc["category_code"] for doc in docs)

Downloaded/evaluable documents: 44


Counter({'H_xai_calibration': 7,
         'B_college_depression_anxiety': 6,
         'F_sleep_wellbeing': 6,
         'I_rag_llm_eval': 6,
         'A_student_mental_health': 5,
         'D_academic_impairment': 4,
         'G_screening_communication': 4,
         'C_loneliness_isolation': 3,
         'E_financial_food': 3})

## Chunk extraction

The chunking is page-aware: each chunk has a page number, which is required for source attribution in a RAG paper. Chunks are word-based because PDF extraction often produces irregular line breaks.

In [4]:
def clean_text(text):
    return re.sub(r"\s+", " ", text or "").strip()

def chunk_words(text, words_per_chunk=WORDS_PER_CHUNK, overlap=CHUNK_OVERLAP):
    words = text.split()
    if not words:
        return []
    chunks = []
    step = max(1, words_per_chunk - overlap)
    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + words_per_chunk])
        if len(chunk) >= 250:
            chunks.append(chunk)
        if start + words_per_chunk >= len(words):
            break
    return chunks

def extract_doc_chunks(doc):
    chunks = []
    try:
        reader = PdfReader(doc["abs_pdf_path"])
    except Exception as exc:
        print("PDF read failed", doc["doc_id"], exc)
        return chunks
    for page_idx, page in enumerate(reader.pages[:MAX_PAGES_PER_DOC], start=1):
        try:
            page_text = clean_text(page.extract_text())
        except Exception:
            page_text = ""
        for chunk_idx, chunk_text in enumerate(chunk_words(page_text), start=1):
            chunks.append({
                "chunk_id": f"{doc['doc_id']}_p{page_idx}_c{chunk_idx}",
                "doc_id": doc["doc_id"],
                "category_code": doc.get("category_code", ""),
                "title": doc.get("title", ""),
                "main_topic": doc.get("main_topic", ""),
                "source_url": doc.get("source_url") or doc.get("pdf_url", ""),
                "page": page_idx,
                "text": chunk_text,
            })
            if len(chunks) >= MAX_CHUNKS_PER_DOC:
                return chunks
    return chunks

chunk_cache = CACHE_DIR / f"chunks_p{MAX_PAGES_PER_DOC}_c{MAX_CHUNKS_PER_DOC}_w{WORDS_PER_CHUNK}.jsonl"
if chunk_cache.exists():
    chunks = [json.loads(line) for line in chunk_cache.read_text(encoding="utf-8").splitlines() if line.strip()]
else:
    chunks = []
    for doc in docs:
        chunks.extend(extract_doc_chunks(doc))
    chunk_cache.write_text("\n".join(json.dumps(item, ensure_ascii=False) for item in chunks), encoding="utf-8")

print("Chunks:", len(chunks))
print("Documents represented:", len({chunk['doc_id'] for chunk in chunks}))
Counter(chunk["category_code"] for chunk in chunks)

Chunks: 338
Documents represented: 43


Counter({'H_xai_calibration': 56,
         'I_rag_llm_eval': 48,
         'B_college_depression_anxiety': 42,
         'A_student_mental_health': 40,
         'F_sleep_wellbeing': 40,
         'D_academic_impairment': 32,
         'G_screening_communication': 32,
         'C_loneliness_isolation': 24,
         'E_financial_food': 24})

## Evaluation queries and relevance labels

Two query sources are used:

1. `collection_queries`: the original category-level search queries used during corpus construction. These test whether retrieval can recover the right expertise area.
2. `challenge_queries`: manually written research questions that reflect likely RAG use cases in the paper.

A document is relevant when its `category_code` matches the expected category. For challenge queries with targeted expected documents, relevance is narrowed to those document IDs when possible.

In [5]:
def load_collection_queries():
    queries = []
    with SEARCH_QUERIES_PATH.open("r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 2:
                queries.append({
                    "query_id": f"collection_{len(queries)+1:03d}",
                    "query": parts[1],
                    "expected_category": parts[0],
                    "expected_doc_ids": None,
                    "source": "collection_query",
                })
    return queries

challenge_queries = [
    {
        "query_id": "challenge_phq9",
        "query": "Which sources support using PHQ-9 or PHQ-4 as screening instruments for college student depression?",
        "expected_category": "B_college_depression_anxiety",
        "expected_doc_ids": {"DOC_B006", "DOC_B007", "DOC_A009"},
        "source": "manual_challenge",
    },
    {
        "query_id": "challenge_gad7",
        "query": "Find evidence on the reliability or validity of GAD-7 for university or college students.",
        "expected_category": "B_college_depression_anxiety",
        "expected_doc_ids": {"DOC_B002", "DOC_B003"},
        "source": "manual_challenge",
    },
    {
        "query_id": "challenge_loneliness",
        "query": "What evidence links loneliness, social support, and depressive symptoms in student or young-adult populations?",
        "expected_category": "C_loneliness_isolation",
        "expected_doc_ids": {"DOC_C001", "DOC_C002", "DOC_C003"},
        "source": "manual_challenge",
    },
    {
        "query_id": "challenge_academic",
        "query": "Retrieve studies about academic stress, academic impairment, procrastination, depression, and anxiety in university students.",
        "expected_category": "D_academic_impairment",
        "expected_doc_ids": {"DOC_D001", "DOC_D002", "DOC_D003", "DOC_D004"},
        "source": "manual_challenge",
    },
    {
        "query_id": "challenge_financial",
        "query": "Find sources about food insecurity, material hardship, financial stress, and student mental health.",
        "expected_category": "E_financial_food",
        "expected_doc_ids": {"DOC_E001", "DOC_E002", "DOC_E003"},
        "source": "manual_challenge",
    },
    {
        "query_id": "challenge_xai",
        "query": "Retrieve sources explaining SHAP, local feature attribution, calibration, and why explanations are not causal claims.",
        "expected_category": "H_xai_calibration",
        "expected_doc_ids": {"DOC_H001", "DOC_H002", "DOC_H003", "DOC_H005"},
        "source": "manual_challenge",
    },
    {
        "query_id": "challenge_rag_eval",
        "query": "Which papers define RAG, citation faithfulness, hallucination evaluation, and factuality checks for LLM generated text?",
        "expected_category": "I_rag_llm_eval",
        "expected_doc_ids": {"DOC_I001", "DOC_I002", "DOC_I003", "DOC_I004", "DOC_I005", "DOC_I006"},
        "source": "manual_challenge",
    },
]

queries = load_collection_queries() + challenge_queries
available_doc_ids = {chunk["doc_id"] for chunk in chunks}
for query in queries:
    if query["expected_doc_ids"]:
        query["expected_doc_ids"] = set(query["expected_doc_ids"]) & available_doc_ids

print("Evaluation queries:", len(queries))
Counter(query["source"] for query in queries)

Evaluation queries: 70


Counter({'collection_query': 63, 'manual_challenge': 7})

## Context construction

`metadata_context` is deterministic and cheap. `ollama_context` is closer to the Claude guide: it generates a short context sentence for each chunk using the source document title/topic/category plus the chunk text.

The Ollama context is cached because local generation can take a long time over the full corpus.

In [6]:
def metadata_context(chunk):
    return (
        f"Document title: {chunk['title']}. "
        f"Category: {chunk['category_code']}. "
        f"Document topic: {chunk['main_topic']}. "
        f"Page: {chunk['page']}."
    )

def context_prompt(chunk):
    return f"""You are preparing a chunk for retrieval in a scientific RAG system.

Document title: {chunk['title']}
Document category: {chunk['category_code']}
Document topic: {chunk['main_topic']}
Page: {chunk['page']}

Chunk:
{chunk['text'][:3500]}

Write one concise sentence, maximum 45 words, that explains what this chunk is about and how it fits the document. Do not add claims not supported by the title, topic, or chunk."""

context_cache_path = CACHE_DIR / f"ollama_contexts_{CONTEXT_MODEL.replace(':', '_')}.json"
if context_cache_path.exists():
    ollama_context_cache = json.loads(context_cache_path.read_text(encoding="utf-8"))
else:
    ollama_context_cache = {}

def ollama_context(chunk):
    cache_key = chunk["chunk_id"]
    if cache_key not in ollama_context_cache:
        text = generate_text(context_prompt(chunk), model=CONTEXT_MODEL, timeout=360)
        ollama_context_cache[cache_key] = text[:700]
        context_cache_path.write_text(json.dumps(ollama_context_cache, indent=2, ensure_ascii=False), encoding="utf-8")
    return ollama_context_cache[cache_key]

def text_for_condition(chunk, condition):
    if condition == "plain":
        return chunk["text"]
    if condition == "metadata_context":
        return metadata_context(chunk) + "\n\n" + chunk["text"]
    if condition == "ollama_context":
        return ollama_context(chunk) + "\n\n" + chunk["text"]
    raise ValueError(condition)

## Dense embedding index with cache

Each condition gets a separate embedding matrix. Embeddings are cached as `.npy` files plus a chunk-id manifest so repeated runs are reproducible and faster.

In [7]:
def cache_stem(condition):
    signature = f"{condition}_{EMBED_MODEL}_{len(chunks)}_{MAX_PAGES_PER_DOC}_{MAX_CHUNKS_PER_DOC}_{WORDS_PER_CHUNK}"
    return CACHE_DIR / re.sub(r"[^A-Za-z0-9_.-]+", "_", signature)

def build_embeddings(condition):
    stem = cache_stem(condition)
    matrix_path = stem.with_suffix(".npy")
    manifest_path = stem.with_suffix(".json")
    chunk_ids = [chunk["chunk_id"] for chunk in chunks]
    if matrix_path.exists() and manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("chunk_ids") == chunk_ids:
            return np.load(matrix_path)
    vectors = []
    start = time.time()
    for index, chunk in enumerate(chunks, start=1):
        vectors.append(embed_text(text_for_condition(chunk, condition)))
        if index % 10 == 0 or index == len(chunks):
            print(f"{condition}: embedded {index}/{len(chunks)} chunks after {time.time() - start:.1f}s")
    matrix = np.vstack(vectors)
    np.save(matrix_path, matrix)
    manifest_path.write_text(json.dumps({"chunk_ids": chunk_ids}, indent=2), encoding="utf-8")
    return matrix

# For a fast pass, start with plain and metadata_context. Add ollama_context for final paper runs.
# Full paper run: plain chunks, deterministic metadata context, and Ollama-generated context.
CONDITIONS_TO_RUN = ["plain", "metadata_context", "ollama_context"]

embedding_matrices = {condition: build_embeddings(condition) for condition in CONDITIONS_TO_RUN}
{condition: matrix.shape for condition, matrix in embedding_matrices.items()}

ollama_context: embedded 10/338 chunks after 65.4s


ollama_context: embedded 20/338 chunks after 125.6s


Ollama call failed on attempt 1/3: HTTP Error 500: Internal Server Error {"error":"the input length exceeds the context length"}


Ollama call failed on attempt 2/3: HTTP Error 500: Internal Server Error {"error":"the input length exceeds the context length"}


Ollama call failed on attempt 3/3: HTTP Error 500: Internal Server Error {"error":"the input length exceeds the context length"}


...

ollama_context: embedded 310/338 chunks after 1945.4s


ollama_context: embedded 320/338 chunks after 2007.1s


ollama_context: embedded 330/338 chunks after 2069.3s


ollama_context: embedded 338/338 chunks after 2116.9s


{'plain': (338, 768),
 'metadata_context': (338, 768),
 'ollama_context': (338, 768)}

## BM25 lexical retrieval

The Anthropic recipe combines contextual embeddings with lexical retrieval/reranking. To keep this notebook local and dependency-light, BM25 is implemented directly. Hybrid retrieval uses a normalized weighted average of dense cosine scores and BM25 scores.

In [8]:
TOKEN_RE = re.compile(r"[a-zA-Z][a-zA-Z0-9_-]+")

def tokenize(text):
    return [token.lower() for token in TOKEN_RE.findall(text)]

class BM25Index:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.doc_tokens = [tokenize(text) for text in documents]
        self.doc_lens = np.asarray([len(tokens) for tokens in self.doc_tokens], dtype=np.float32)
        self.avgdl = float(self.doc_lens.mean()) if len(self.doc_lens) else 1.0
        df = Counter()
        for tokens in self.doc_tokens:
            df.update(set(tokens))
        n_docs = len(self.doc_tokens)
        self.idf = {term: math.log(1 + (n_docs - freq + 0.5) / (freq + 0.5)) for term, freq in df.items()}
        self.term_freqs = [Counter(tokens) for tokens in self.doc_tokens]

    def scores(self, query):
        query_terms = tokenize(query)
        scores = np.zeros(len(self.doc_tokens), dtype=np.float32)
        for term in query_terms:
            idf = self.idf.get(term)
            if idf is None:
                continue
            for idx, freqs in enumerate(self.term_freqs):
                tf = freqs.get(term, 0)
                if tf == 0:
                    continue
                denom = tf + self.k1 * (1 - self.b + self.b * self.doc_lens[idx] / self.avgdl)
                scores[idx] += idf * (tf * (self.k1 + 1) / denom)
        return scores

def normalize(scores):
    scores = np.asarray(scores, dtype=np.float32)
    lo, hi = float(scores.min()), float(scores.max())
    if hi - lo < 1e-8:
        return np.zeros_like(scores)
    return (scores - lo) / (hi - lo)

bm25_indexes = {
    condition: BM25Index([text_for_condition(chunk, condition) for chunk in chunks])
    for condition in CONDITIONS_TO_RUN
}
print("BM25 indexes:", list(bm25_indexes))

BM25 indexes: ['plain', 'metadata_context', 'ollama_context']


## Retrieval and metric functions

In [9]:
def cosine_scores(query_vector, matrix):
    query_norm = np.linalg.norm(query_vector) or 1.0
    matrix_norm = np.linalg.norm(matrix, axis=1)
    matrix_norm[matrix_norm == 0] = 1.0
    return matrix @ query_vector / (matrix_norm * query_norm)

query_embedding_cache = {}
def query_embedding(query):
    key = hashlib.sha1((EMBED_MODEL + "\n" + query).encode("utf-8")).hexdigest()
    if key not in query_embedding_cache:
        query_embedding_cache[key] = embed_text(query)
    return query_embedding_cache[key]

def retrieve(query, condition, mode, top_k=TOP_K, dense_weight=0.70):
    dense = cosine_scores(query_embedding(query), embedding_matrices[condition])
    if mode == "dense":
        scores = dense
    elif mode == "hybrid":
        lexical = bm25_indexes[condition].scores(query)
        scores = dense_weight * normalize(dense) + (1 - dense_weight) * normalize(lexical)
    else:
        raise ValueError(mode)
    order = np.argsort(-scores)[:top_k]
    return [(int(idx), float(scores[idx]), chunks[int(idx)]) for idx in order]

def relevant_doc_ids(query_spec):
    if query_spec.get("expected_doc_ids"):
        return set(query_spec["expected_doc_ids"])
    return {doc["doc_id"] for doc in docs if doc.get("category_code") == query_spec["expected_category"]}

def dcg(relevances):
    return sum(rel / math.log2(rank + 2) for rank, rel in enumerate(relevances))

def evaluate_one(query_spec, condition, mode, top_k=TOP_K):
    expected_docs = relevant_doc_ids(query_spec)
    results = retrieve(query_spec["query"], condition, mode, top_k=top_k)
    retrieved_docs = [item[2]["doc_id"] for item in results]
    retrieved_categories = [item[2]["category_code"] for item in results]
    binary_relevance = [1 if doc_id in expected_docs else 0 for doc_id in retrieved_docs]
    first_hit = next((idx + 1 for idx, rel in enumerate(binary_relevance) if rel), None)
    ideal_rels = sorted(binary_relevance, reverse=True)
    idcg = dcg(ideal_rels) or 1.0
    return {
        "query_id": query_spec["query_id"],
        "source": query_spec["source"],
        "condition": condition,
        "mode": mode,
        "expected_category": query_spec["expected_category"],
        "recall_at_k": 1.0 if first_hit else 0.0,
        "mrr_at_k": 1.0 / first_hit if first_hit else 0.0,
        "ndcg_at_k": dcg(binary_relevance) / idcg,
        "category_accuracy_at_1": 1.0 if retrieved_categories and retrieved_categories[0] == query_spec["expected_category"] else 0.0,
        "first_hit_rank": first_hit or "",
        "top_doc_ids": ",".join(retrieved_docs),
        "top_categories": ",".join(retrieved_categories),
        "query": query_spec["query"],
    }

def summarize(rows):
    grouped = defaultdict(list)
    for row in rows:
        grouped[(row["condition"], row["mode"], row["source"])].append(row)
    summary = []
    for (condition, mode, source), items in sorted(grouped.items()):
        summary.append({
            "condition": condition,
            "mode": mode,
            "source": source,
            "n_queries": len(items),
            "recall_at_k": round(float(np.mean([item["recall_at_k"] for item in items])), 4),
            "mrr_at_k": round(float(np.mean([item["mrr_at_k"] for item in items])), 4),
            "ndcg_at_k": round(float(np.mean([item["ndcg_at_k"] for item in items])), 4),
            "category_accuracy_at_1": round(float(np.mean([item["category_accuracy_at_1"] for item in items])), 4),
        })
    return summary

## Run evaluation

In [10]:
rows = []
for condition in CONDITIONS_TO_RUN:
    for mode in ["dense", "hybrid"]:
        for query_spec in queries:
            rows.append(evaluate_one(query_spec, condition, mode, top_k=TOP_K))

summary = summarize(rows)
summary

[{'condition': 'metadata_context',
  'mode': 'dense',
  'source': 'collection_query',
  'n_queries': 63,
  'recall_at_k': 0.6508,
  'mrr_at_k': 0.6164,
  'ndcg_at_k': 0.6208,
  'category_accuracy_at_1': 0.5873},
 {'condition': 'metadata_context',
  'mode': 'dense',
  'source': 'manual_challenge',
  'n_queries': 7,
  'recall_at_k': 1.0,
  'mrr_at_k': 0.9286,
  'ndcg_at_k': 0.9634,
  'category_accuracy_at_1': 0.8571},
 {'condition': 'metadata_context',
  'mode': 'hybrid',
  'source': 'collection_query',
  'n_queries': 63,
  'recall_at_k': 0.746,
  'mrr_at_k': 0.6783,
  'ndcg_at_k': 0.6895,
  'category_accuracy_at_1': 0.6508},
 {'condition': 'metadata_context',
  'mode': 'hybrid',
  'source': 'manual_challenge',
  'n_queries': 7,
  'recall_at_k': 1.0,
  'mrr_at_k': 1.0,
  'ndcg_at_k': 0.9976,
  'category_accuracy_at_1': 1.0},
 {'condition': 'ollama_context',
  'mode': 'dense',
  'source': 'collection_query',
  'n_queries': 63,
  'recall_at_k': 0.6825,
  'mrr_at_k': 0.5513,
  'ndcg_at_k': 

## Error analysis table

For a scientific defense, do not report only aggregate metrics. Inspect misses and category confusions, because they explain whether contextualization actually helps the RAG use case.

In [11]:
misses = [row for row in rows if row["recall_at_k"] == 0.0]
print("Misses:", len(misses))
misses[:15]

Misses: 102


[{'query_id': 'collection_007',
  'source': 'collection_query',
  'condition': 'plain',
  'mode': 'dense',
  'expected_category': 'B_college_depression_anxiety',
  'recall_at_k': 0.0,
  'mrr_at_k': 0.0,
  'ndcg_at_k': 0.0,
  'category_accuracy_at_1': 0.0,
  'first_hit_rank': '',
  'top_doc_ids': 'DOC_A001,DOC_A001,DOC_A001,DOC_A002,DOC_A001',
  'top_categories': 'A_student_mental_health,A_student_mental_health,A_student_mental_health,A_student_mental_health,A_student_mental_health',
  'query': 'depression anxiety prevalence college students cross-sectional pdf'},
 {'query_id': 'collection_010',
  'source': 'collection_query',
  'condition': 'plain',
  'mode': 'dense',
  'expected_category': 'B_college_depression_anxiety',
  'recall_at_k': 0.0,
  'mrr_at_k': 0.0,
  'ndcg_at_k': 0.0,
  'category_accuracy_at_1': 0.0,
  'first_hit_rank': '',
  'top_doc_ids': 'DOC_A001,DOC_A002,DOC_F002,DOC_A002,DOC_A001',
  'top_categories': 'A_student_mental_health,A_student_mental_health,F_sleep_wellbein

## Persist results

The output files are suitable for inclusion in a paper appendix or for later statistical analysis.

In [12]:
timestamp = time.strftime("%Y%m%d_%H%M%S")
result_json = LOG_DIR / f"paper_contextual_retrieval_eval_{timestamp}.json"
detail_csv = LOG_DIR / f"paper_contextual_retrieval_eval_{timestamp}_details.csv"
summary_csv = LOG_DIR / f"paper_contextual_retrieval_eval_{timestamp}_summary.csv"

result_json.write_text(json.dumps({
    "settings": {
        "embed_model": EMBED_MODEL,
        "context_model": CONTEXT_MODEL,
        "max_pages_per_doc": MAX_PAGES_PER_DOC,
        "max_chunks_per_doc": MAX_CHUNKS_PER_DOC,
        "words_per_chunk": WORDS_PER_CHUNK,
        "chunk_overlap": CHUNK_OVERLAP,
        "top_k": TOP_K,
        "conditions": CONDITIONS_TO_RUN,
    },
    "corpus": {
        "documents": len(docs),
        "chunks": len(chunks),
        "categories": dict(Counter(chunk["category_code"] for chunk in chunks)),
    },
    "summary": summary,
    "details": rows,
}, indent=2, ensure_ascii=False), encoding="utf-8")

with detail_csv.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

with summary_csv.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary[0].keys()))
    writer.writeheader()
    writer.writerows(summary)

print(result_json)
print(detail_csv)
print(summary_csv)

C:\Users\MossabBatal\Desktop\psychology_mas\knowledge_db\logs\paper_contextual_retrieval_eval_20260630_013954.json
C:\Users\MossabBatal\Desktop\psychology_mas\knowledge_db\logs\paper_contextual_retrieval_eval_20260630_013954_details.csv
C:\Users\MossabBatal\Desktop\psychology_mas\knowledge_db\logs\paper_contextual_retrieval_eval_20260630_013954_summary.csv


## How to report this in the paper

Suggested wording:

> We evaluated retrieval over a local corpus of downloaded psychology, XAI, and RAG-evaluation PDFs. Each PDF was parsed into page-aware chunks, and each chunk retained document ID, page number, title, source URL, and expertise category. We compared plain dense retrieval against contextual dense retrieval, where each chunk was prepended with either deterministic document metadata or a locally generated context sentence from an Ollama-hosted language model. Retrieval was evaluated with Recall@5, MRR@5, nDCG@5, and top-1 category accuracy over category-level corpus-construction queries and manually written challenge queries representing expected system use cases.

Defensible interpretation rules:

- If contextual retrieval improves `MRR@5` but not `Recall@5`, say it improves ranking, not coverage.
- If hybrid retrieval improves over dense retrieval, say lexical matching remains important for acronyms such as PHQ-9, GAD-7, SHAP, and RAG.
- If collection queries perform better than manual challenge queries, report that the internal benchmark is easier and partly aligned with corpus acquisition.
- If Ollama-generated context does not improve over metadata context, the safer conclusion is that structured metadata is already a strong local substitute for generated context in this corpus.
- Do not claim clinical validity from retrieval metrics; this only evaluates evidence retrieval quality.